# Piloto TrashNet: ¿nos quedamos con este dataset?

Author: Anyely Sanjuan & Juan Díaz

**Objetivo.** Decidir con evidencia si TrashNet es viable para el proyecto. Medimos cuatro cosas:
1. **Tiempo** por época en *esta* máquina y proyección del presupuesto (≤ 2 h para la corrida final).
2. **Aprendibilidad**: si una CNN pequeña supera con claridad a un baseline trivial.
3. **Variabilidad** entre semillas (insumo para calcular el número de ejecuciones $R$ del preregistro).
4. **Calidad de datos**: archivos corruptos, duplicados y desbalance.

**Reglas metodológicas de este piloto**
- El conjunto de **test se crea y se congela** (huella SHA-256), pero **no se evalúa aquí**. Todo se mide en *validación*.
- El preprocesamiento no usa estadísticas del dataset (solo divide entre 255), así que no hay fuga de información.
- La augmentación ocurre *dentro del modelo* y solo durante el entrenamiento.

**Cómo ejecutarlo:** kernel `.venv` → *Run All*. La primera vez descarga unos 43 MB.
Datos: [Hugging Face `garythung/trashnet`](https://huggingface.co/datasets/garythung/trashnet), licencia MIT. Hay que citar [github.com/garythung/trashnet](https://github.com/garythung/trashnet).

## 0. Criterios de decisión (declararlos ANTES de ejecutar)

**Fecha y hora de la declaración:** _(completar)_ · **Máquina:** _(CPU de … / GPU de …)_

| Criterio | Verde | Amarillo | Rojo |
|---|---|---|---|
| Tiempo mediano por época, CNN (sin contar la 1.ª) | ≤ 8 s | 8–12 s | > 12 s |
| Proyección: 3 modelos × 10 semillas × 20 épocas | ≤ 1.5 h | 1.5–2 h | > 2 h |
| Macro-F1 en validación (CNN) | ≥ 0.50 | 0.30–0.50 | < 0.30 |
| Margen de mejora para E3 (macro-F1 < 0.90) | sí | — | no |
| Casi-duplicados entre clases distintas | 0 | 1–5 | > 5 |

Los umbrales son una **propuesta**: acuérdenlos entre los dos antes de mirar resultados. Cambiarlos después invalida el piloto.
La decisión de tiempo vale para la **máquina donde se harán las corridas finales**. Los tiempos de CPU y GPU no son comparables.

In [ ]:
# === Configuración del piloto (se guarda junto con los resultados) ===
import time
T0_NOTEBOOK = time.perf_counter()

CFG = {
    "img_size": 64,          # lado de la imagen cuadrada (px). Empieza en 64; si va sobrado, prueba 96 o 128
    "batch_size": 32,
    "epochs": 20,
    "patience": 5,           # early stopping sobre val_loss
    "learning_rate": 1e-3,
    "seeds": [0],            # piloto rápido: [0]. Para estimar varianza: [0, 1, 2, 3, 4]
    "split_seed": 1234,      # semilla de la PARTICIÓN (fija; no cambia entre semillas de entrenamiento)
    "frac_train": 0.70,
    "frac_val": 0.15,        # test = 1 - 0.70 - 0.15 = 0.15
    "near_dup_max_dist": 2,  # distancia Hamming máxima (de 64 bits) para considerar casi-duplicado
    "augment": True,         # flip horizontal + rotación pequeña, solo en entrenamiento
    "class_weights": False,  # compensar el desbalance (trash ≈ 5 %); se decide en el preregistro
    "deterministic": False,  # True = más reproducible y más lento
    "timing_others": True,   # cronometra 3 épocas de un MLP y de MobileNetV2 congelada (solo tiempo)
    "budget": {"models": 3, "seeds": 10, "epochs": 20, "hours_max": 2.0},
}

In [ ]:
# Rutas e imports (esta parte NO necesita TensorFlow)
import sys, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("No encuentro la raíz del repo (requirements.txt + src/). Abre el notebook desde el repo.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import data_trashnet as dtn

RAW_DIR   = ROOT / "data" / "raw" / "trashnet"          # ignorado por git
ZIP_PATH  = RAW_DIR / "dataset-resized.zip"
INTERIM   = ROOT / "data" / "interim"                   # ignorado por git (caché)
SPLIT_CSV = ROOT / "data" / "splits" / f"trashnet_split_s{CFG['split_seed']}.csv"  # SÍ se versiona
SUMS      = ROOT / "data" / "SHA256SUMS.txt"            # SÍ se versiona
RUN_ID    = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
OUT       = ROOT / "results" / "pilot" / RUN_ID         # se versionan csv/json/png
OUT.mkdir(parents=True, exist_ok=True)
print("Raíz del repo:", ROOT)
print("Resultados de esta corrida:", OUT.relative_to(ROOT))

## 1. Descarga y verificación

Pasos: se descarga el zip (unos 43 MB) a `data/raw/trashnet/`, se calcula su SHA-256 y se registra en `data/SHA256SUMS.txt`. Ese registro va al repositorio, así que cuando tu compañera ejecute el notebook se comprobará que tiene **exactamente el mismo archivo**. Después se descomprime, verificando el CRC de cada imagen.

Si la descarga falla (por un proxy o un firewall), descarga a mano el enlace que muestra el error, guárdalo con la ruta indicada y vuelve a ejecutar la celda.

In [ ]:
t = time.perf_counter()
dtn.download(ZIP_PATH)
zip_sha = dtn.sha256_file(ZIP_PATH)
status = dtn.register_checksum(SUMS, "data/raw/trashnet/dataset-resized.zip", zip_sha)
print("SHA-256 del zip:", zip_sha, "->", status)
if status == "DIFERENTE":
    raise RuntimeError("El zip NO coincide con el registrado en data/SHA256SUMS.txt (¿otra versión?). No continuar.")
try:
    CLASS_ROOT = dtn.find_class_root(RAW_DIR)
    print("Ya estaba descomprimido en:", CLASS_ROOT.relative_to(ROOT))
except FileNotFoundError:
    CLASS_ROOT = dtn.extract(ZIP_PATH, RAW_DIR)
    print("Descomprimido en:", CLASS_ROOT.relative_to(ROOT))
print(f"[{time.perf_counter() - t:.1f} s]")

## 2. EDA general: índice, integridad y desbalance

**¿Se "tokeniza" la imagen?** No. *Tokenizar* es partir texto en unidades (tokens). Una imagen ya es numérica: un tensor de forma $(H, W, 3)$ con enteros de 0 a 255 (rojo, verde, azul). Lo que hacemos es *decodificar* el JPEG a ese tensor, redimensionarlo y escalarlo a $[0,1]$ (sección 4).

En esta sección se abre cada archivo para verificar que no esté corrupto. También se calcula su SHA-256, que detecta duplicados exactos, y un *dHash* de 64 bits que detecta casi-duplicados. El índice se guarda en caché en `data/interim/`.

In [ ]:
t = time.perf_counter()
INDEX_CSV = INTERIM / "trashnet_index.csv"
if INDEX_CSV.exists():
    idx = dtn.read_csv(INDEX_CSV)
    idx["ok"] = idx["ok"].astype(str).eq("True")
    print("Índice leído de caché:", INDEX_CSV.relative_to(ROOT))
else:
    idx = dtn.build_index(CLASS_ROOT)
    INTERIM.mkdir(parents=True, exist_ok=True)
    idx.to_csv(INDEX_CSV, index=False)
bad = idx[~idx["ok"]]
print(f"{len(idx)} archivos | {len(bad)} corruptos  [{time.perf_counter() - t:.1f} s]")
if len(bad):
    display(bad[["relpath", "error"]])
ok = idx[idx["ok"]].reset_index(drop=True).astype({"width": int, "height": int})  # solo imágenes válidas

In [ ]:
counts = ok["label"].value_counts().reindex(dtn.CLASSES)
tab = pd.DataFrame({"n": counts, "esperado (README)": pd.Series(dtn.EXPECTED_COUNTS),
                    "%": (100 * counts / counts.sum()).round(1)})
tab["coincide"] = tab["n"] == tab["esperado (README)"]
display(tab)
print("Total:", int(counts.sum()), "| razón de desbalance máx/mín:", round(counts.max() / counts.min(), 2))
print("Tamaños (ancho, alto):", ok.groupby(["width", "height"]).size().to_dict())
print("Modos de color:", ok["mode"].value_counts().to_dict())

fig, ax = plt.subplots(figsize=(6, 3))
counts.plot.bar(ax=ax, color="#4C78A8")
ax.set_ylabel("imágenes"); ax.set_title("Imágenes por clase"); ax.tick_params(axis="x", rotation=0)
fig.tight_layout(); fig.savefig(OUT / "eda_clases.png", dpi=120); plt.show()

### Duplicados y casi-duplicados

Si dos fotos casi iguales caen una en train y otra en test, el modelo "ya vio" la respuesta y la métrica queda inflada. A eso se le llama **fuga** (*data leakage*). Para evitarlo, las imágenes a distancia Hamming $\le$ `near_dup_max_dist` se agrupan, y **cada grupo va completo a una sola partición**.

**Revisa visualmente los pares que aparecen abajo.** Si no son la misma foto, son falsos positivos: baja el umbral a 0 o 1. Un grupo de más de 20 imágenes indica encadenamiento de falsos positivos.

In [ ]:
t = time.perf_counter()
pairs = dtn.near_duplicate_pairs(ok["dhash"].tolist(), max_dist=CFG["near_dup_max_dist"])
ok["group"] = dtn.assign_groups(ok, pairs)
dups = dtn.duplicate_report(ok, pairs)
gsz = ok["group"].value_counts()
n_exact = int(ok["sha256"].duplicated().sum())
n_cross = int(dups["cross_class"].sum()) if len(dups) else 0
print(f"Duplicados exactos (mismo SHA-256): {n_exact}")
print(f"Pares casi-duplicados (dist ≤ {CFG['near_dup_max_dist']}): {len(pairs)} | entre clases distintas: {n_cross}")
print(f"{ok['group'].nunique()} grupos para {len(ok)} imágenes | grupo más grande: {gsz.max()}  [{time.perf_counter() - t:.1f} s]")
if gsz.max() > 20:
    print("AVISO: grupo muy grande -> probable encadenamiento de falsos positivos; baja near_dup_max_dist.")
display(dups.head(10))

if len(dups):
    show = dups.head(4)
    fig, axes = plt.subplots(len(show), 2, figsize=(5, 2.2 * len(show)), squeeze=False)
    for k, r in enumerate(show.itertuples()):
        for c, rel in enumerate([r.relpath_a, r.relpath_b]):
            axes[k, c].imshow(plt.imread(CLASS_ROOT / rel)); axes[k, c].axis("off")
            axes[k, c].set_title(f"{rel}  (d={r.dist})", fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "eda_duplicados.png", dpi=120); plt.show()

## 3. Partición estratificada por grupos y test congelado

La partición es 70/15/15, estratificada por clase, con `split_seed` fija. **La primera ejecución crea** `data/splits/trashnet_split_s1234.csv` y registra la huella del test. **Las siguientes ejecuciones la cargan** y verifican que no haya cambiado.

**Importante:** quien la cree primero debe hacer *commit* de ese CSV y de `SHA256SUMS.txt`, y la otra persona debe hacer `git pull` **antes** de ejecutar. Así las dos usan la misma partición.

In [ ]:
split, created = dtn.load_or_create_split(ok, SPLIT_CSV, seed=CFG["split_seed"],
                                         frac_train=CFG["frac_train"], frac_val=CFG["frac_val"])
test_fp = dtn.split_fingerprint(split, "test")
st = dtn.register_checksum(SUMS, f"test_set:{SPLIT_CSV.relative_to(ROOT).as_posix()}", test_fp)
print("Partición", "CREADA ahora" if created else "cargada (congelada)", "->", SPLIT_CSV.relative_to(ROOT))
print("Huella SHA-256 del test:", test_fp, "->", st)
if st == "DIFERENTE":
    raise RuntimeError("La huella del test cambió: la partición no es la registrada. No continuar.")

# Pruebas mínimas de no-fuga
assert split.groupby("group")["split"].nunique().max() == 1, "Un grupo quedó repartido entre particiones"
assert not set(split.loc[split.split == "test", "relpath"]) & set(split.loc[split.split != "test", "relpath"])
display(pd.crosstab(split["label"], split["split"], margins=True)[["train", "val", "test", "All"]])

### EDA visual (solo con imágenes de *train*)

Las imágenes se miran solo en *train*, para no "estudiar" el test.

In [ ]:
tr = split[split["split"] == "train"]
ncol = 5
fig, axes = plt.subplots(len(dtn.CLASSES), ncol, figsize=(9, 1.6 * len(dtn.CLASSES) + 0.5), squeeze=False)
for i, c in enumerate(dtn.CLASSES):
    sub = tr[tr["label"] == c]
    sample = sub.sample(min(ncol, len(sub)), random_state=0)
    for j in range(ncol):
        axes[i, j].axis("off")
    for j, rel in enumerate(sample["relpath"]):
        axes[i, j].imshow(plt.imread(CLASS_ROOT / rel))
    axes[i, 0].set_title(c, loc="left", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "eda_muestras.png", dpi=110); plt.show()

stats = tr[["relpath", "label"]].merge(ok[["relpath", "bright_frac", "mean_r", "mean_g", "mean_b"]], on="relpath")
display(stats.groupby("label")[["bright_frac", "mean_r", "mean_g", "mean_b"]].mean().round(2))
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([stats.loc[stats.label == c, "bright_frac"] for c in dtn.CLASSES])
ax.set_xticks(range(1, len(dtn.CLASSES) + 1), dtn.CLASSES)
ax.set_ylabel("fracción de píxeles ~blancos"); ax.set_title("Fondo blanco por clase (train)")
fig.tight_layout(); fig.savefig(OUT / "eda_fondo.png", dpi=120); plt.show()

### Cierre del EDA (borrador: corrígelo con lo que veas arriba)

- **Decisión de preprocesamiento:** redimensionar de 512×384 a S×S. Esto deforma la relación 4:3, pero igual en todo el pipeline. Luego escalar a $[0,1]$ dividiendo entre 255, sin normalizar con estadísticas del dataset. Augmentación ligera (flip y rotación) solo en train.
- **Impacto en la arquitectura:** el fondo es blanco y homogéneo, y la iluminación es controlada. Hay riesgo de que el modelo aprenda *atajos* (brillo o fondo) en vez de la forma o el material. La clase `trash` es minoritaria (~5 %), por eso la métrica primaria debe ser macro (macro-F1 o balanced accuracy) y no la accuracy.
- **Riesgos de fuga:** casi-duplicados → agrupados antes de partir. Test → congelado con huella. Augmentación → después de partir y solo en train. Normalización → sin estadísticas globales.

## 4. Baselines triviales (el piso de referencia)

Cualquier modelo debe superar con claridad a estos dos clasificadores triviales:
- **Mayoritario:** siempre predice la clase más frecuente.
- **Estratificado:** predice al azar respetando las proporciones de cada clase.

Si una CNN no los supera, el problema está en el pipeline, no en el dataset.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score)

def metrics(y, p) -> dict:
    return {"accuracy": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, p))}

Y = {p: split.loc[split["split"] == p, "label_id"].to_numpy() for p in ["train", "val"]}
dummy = {}
for strat in ["most_frequent", "stratified"]:
    clf = DummyClassifier(strategy=strat, random_state=0).fit(np.zeros((len(Y["train"]), 1)), Y["train"])
    dummy[strat] = metrics(Y["val"], clf.predict(np.zeros((len(Y["val"]), 1))))
display(pd.DataFrame(dummy).T.round(3))

## 5. TensorFlow, dispositivo y pipeline `tf.data`

`src/device.py` usa la GPU si existe y si no la CPU, fija las semillas y devuelve información de versiones y hardware, que se guarda con los resultados.

**De archivo a tensor** (lo que en texto sería "tokenizar"):

$$\text{bytes JPEG} \xrightarrow{\text{decode}} \underbrace{(384, 512, 3)}_{\text{uint8}} \xrightarrow{\text{resize}} \underbrace{(S, S, 3)}_{\text{float32},\,[0,255]} \xrightarrow{/255} [0,1] \xrightarrow{\text{batch}} (B, S, S, 3)$$

Las etiquetas son enteros $y \in \{0,\dots,5\}$ con forma $(B,)$. El `.cache()` guarda las imágenes ya decodificadas en RAM, así que **la 1.ª época es más lenta** (lee los JPEG del disco) y por eso se excluye del tiempo mediano.

In [ ]:
import tensorflow as tf
import keras
from keras import layers
from src.device import setup_device

DEVICE = setup_device(seed=CFG["seeds"][0], deterministic=CFG["deterministic"])
print(json.dumps(DEVICE, indent=2))
(OUT / "device.json").write_text(json.dumps(DEVICE, indent=2), encoding="utf-8")
(OUT / "config.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
S, B = CFG["img_size"], CFG["batch_size"]

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)  # uint8   (H, W, 3)
    img = tf.image.resize(img, [S, S], antialias=True)                  # float32 (S, S, 3) en [0, 255]
    return img / 255.0, label                                           # float32 (S, S, 3) en [0, 1]

def base_ds(part: str) -> tf.data.Dataset:
    d = split[split["split"] == part]                    # mismo orden que Y[part]
    paths = [str(CLASS_ROOT / r) for r in d["relpath"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, d["label_id"].astype("int32").to_numpy()))
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()

BASE = {p: base_ds(p) for p in ["train", "val"]}       # el test NO se construye en este piloto
N = {p: len(Y[p]) for p in Y}

def batches(part: str, shuffle: bool, seed: int | None = None) -> tf.data.Dataset:
    ds = BASE[part]
    if shuffle:
        ds = ds.shuffle(N[part], seed=seed, reshuffle_each_iteration=True)
    return ds.batch(B).prefetch(AUTOTUNE)

# Comprobación de formas con 4 imágenes (sin tocar la caché)
xb = tf.stack([load_image(str(CLASS_ROOT / r), 0)[0] for r in split.loc[split.split == "train", "relpath"][:4]])
print("Lote de prueba X:", tuple(xb.shape), xb.dtype.name,
      f"rango [{float(tf.reduce_min(xb)):.2f}, {float(tf.reduce_max(xb)):.2f}]")
print("Tamaños: train =", N["train"], "| val =", N["val"], "| pasos por época =", -(-N["train"] // B))

## 6. Modelo: CNN pequeña (entrenada desde cero)

Son tres bloques `Conv(3×3) → BatchNorm → ReLU → MaxPool`, cada uno con el doble de filtros. Cada bloque reduce a la mitad el alto y el ancho y aumenta los canales. Luego vienen un `GlobalAveragePooling`, que promedia cada mapa de activación a un número, y una capa densa con *softmax* que da 6 probabilidades.

- **Sesgo inductivo:** las convoluciones suponen localidad (los píxeles cercanos se relacionan) y que el mismo patrón importa en cualquier posición (equivarianza a traslación).
- **Pérdida:** entropía cruzada, $\mathcal{L} = -\frac{1}{B}\sum_{i} \log \hat p_{i, y_i}$, donde $\hat p_{i,y_i}$ es la probabilidad que el modelo asigna a la clase correcta de la imagen $i$.
- **Early stopping:** detiene el entrenamiento si `val_loss` no mejora en `patience` épocas y recupera los mejores pesos.

In [ ]:
def build_cnn(seed: int) -> keras.Model:
    aug = ([layers.RandomFlip("horizontal", seed=seed), layers.RandomRotation(0.05, seed=seed)]
           if CFG["augment"] else [])

    def block(f):
        return [layers.Conv2D(f, 3, padding="same", use_bias=False),
                layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D()]

    return keras.Sequential([
        keras.Input((S, S, 3)),            # (B, S, S, 3)
        *aug,                              # activa solo dentro de fit()
        *block(32),                        # (B, S/2, S/2, 32)
        *block(64),                        # (B, S/4, S/4, 64)
        *block(128),                       # (B, S/8, S/8, 128)
        layers.GlobalAveragePooling2D(),   # (B, 128)
        layers.Dropout(0.3),
        layers.Dense(len(dtn.CLASSES), activation="softmax"),  # (B, 6) probabilidades
    ], name="cnn_pequena")

class EpochTimer(keras.callbacks.Callback):
    # Tiempo real de cada época (entrenamiento + validación)
    def on_train_begin(self, logs=None):
        self.times = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t)

build_cnn(0).summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
CW = None
if CFG["class_weights"]:
    w = compute_class_weight("balanced", classes=np.arange(len(dtn.CLASSES)), y=Y["train"])
    CW = dict(enumerate(w))

runs, preds = [], {}
for seed in CFG["seeds"]:
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = build_cnn(seed)
    model.compile(optimizer=keras.optimizers.Adam(CFG["learning_rate"]),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    timer = EpochTimer()
    cbs = [timer, keras.callbacks.EarlyStopping(monitor="val_loss", patience=CFG["patience"],
                                                restore_best_weights=True)]
    t = time.perf_counter()
    hist = model.fit(batches("train", True, seed), validation_data=batches("val", False),
                     epochs=CFG["epochs"], callbacks=cbs, class_weight=CW, verbose=2)
    train_time = time.perf_counter() - t

    p_val = model.predict(batches("val", False), verbose=0).argmax(axis=1)
    p_tr = model.predict(batches("train", False), verbose=0).argmax(axis=1)  # sin augmentación
    h = pd.DataFrame(hist.history)
    h.insert(0, "epoch", np.arange(1, len(h) + 1)); h["sec"] = timer.times
    h.to_csv(OUT / f"history_cnn_seed{seed}.csv", index=False)
    steady = timer.times[1:] or timer.times
    r = {"seed": seed, "params": int(model.count_params()), "epochs_run": len(h),
         "best_epoch": int(h["val_loss"].idxmin()) + 1, "train_time_s": train_time,
         "first_epoch_s": timer.times[0], "median_epoch_s": float(np.median(steady)),
         "train": metrics(Y["train"], p_tr), "val": metrics(Y["val"], p_val)}
    runs.append(r); preds[seed] = p_val
    print(f"seed {seed}: val macro-F1 = {r['val']['macro_f1']:.3f} | val acc = {r['val']['accuracy']:.3f} | "
          f"{r['median_epoch_s']:.1f} s/época (1.ª: {r['first_epoch_s']:.1f} s) | total {train_time:.0f} s")

## 7. Curvas de aprendizaje, métricas y matriz de confusión

**Cómo leerlas:**
- *train* cae y *val* sube → **overfitting**.
- Las dos curvas altas y planas → **underfitting** (el modelo es pequeño o la tasa de aprendizaje es mala).
- Al inicio, la accuracy de *train* puede quedar **por debajo** de la de *val*. No es un error: en train hay augmentación y dropout activos, y en val no.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for r in runs:
    h = pd.read_csv(OUT / f"history_cnn_seed{r['seed']}.csv")
    axes[0].plot(h.epoch, h.loss, label=f"train s{r['seed']}")
    axes[0].plot(h.epoch, h.val_loss, "--", label=f"val s{r['seed']}")
    axes[1].plot(h.epoch, h.accuracy, label=f"train s{r['seed']}")
    axes[1].plot(h.epoch, h.val_accuracy, "--", label=f"val s{r['seed']}")
    axes[2].plot(h.epoch, h.sec, marker="o", label=f"s{r['seed']}")
for ax, t_, yl in zip(axes, ["Pérdida (entropía cruzada)", "Accuracy", "Tiempo por época"],
                      ["loss", "accuracy", "segundos"]):
    ax.set_title(t_); ax.set_xlabel("época"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7); axes[2].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "curvas_cnn.png", dpi=120); plt.show()

res = pd.DataFrame([{"seed": r["seed"], "épocas": r["epochs_run"], "mejor época": r["best_epoch"],
                     **{f"train_{k}": v for k, v in r["train"].items()},
                     **{f"val_{k}": v for k, v in r["val"].items()},
                     "s/época": r["median_epoch_s"], "total s": r["train_time_s"]} for r in runs])
display(res.round(3))

In [ ]:
seed_show = runs[-1]["seed"]
cm = confusion_matrix(Y["val"], preds[seed_show], labels=list(range(6)), normalize="true")
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6), dtn.CLASSES, rotation=45, ha="right"); ax.set_yticks(range(6), dtn.CLASSES)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("predicha"); ax.set_ylabel("real"); ax.set_title(f"Matriz de confusión (val, seed {seed_show}, por fila)")
fig.colorbar(im, fraction=0.046); fig.tight_layout(); fig.savefig(OUT / "confusion_val.png", dpi=120); plt.show()
print(classification_report(Y["val"], preds[seed_show], labels=list(range(6)), target_names=dtn.CLASSES, digits=3, zero_division=0))

## 8. Tiempo de otros candidatos (solo tiempo, no rendimiento)

E1 exige 3 modelos base con **sesgos inductivos distintos**, y la elección final sale de la revisión de literatura. Aquí solo se cronometran dos candidatos típicos para validar el presupuesto:
- **MLP:** sin sesgo espacial; aplana la imagen.
- **MobileNetV2 congelada:** *transfer learning* desde ImageNet. La 1.ª vez descarga unos 9 MB de pesos.

Se entrenan 3 épocas, y **su accuracy no se interpreta**.

In [ ]:
def build_mlp() -> keras.Model:
    return keras.Sequential([keras.Input((S, S, 3)), layers.Flatten(),           # (B, S*S*3)
                             layers.Dense(256, activation="relu"), layers.Dropout(0.3),
                             layers.Dense(len(dtn.CLASSES), activation="softmax")], name="mlp")

def build_mobilenet() -> keras.Model:
    base = keras.applications.MobileNetV2(input_shape=(S, S, 3), include_top=False,
                                          weights="imagenet", pooling="avg")
    base.trainable = False
    return keras.Sequential([keras.Input((S, S, 3)),
                             layers.Rescaling(2.0, offset=-1.0),  # [0,1] -> [-1,1], lo que espera MobileNetV2
                             base, layers.Dropout(0.3),
                             layers.Dense(len(dtn.CLASSES), activation="softmax")], name="mobilenetv2_congelada")

timing = {"cnn_pequena": float(np.median([r["median_epoch_s"] for r in runs]))}
params = {"cnn_pequena": runs[0]["params"]}
if CFG["timing_others"]:
    for name, fn in [("mlp", build_mlp), ("mobilenetv2_congelada", build_mobilenet)]:
        keras.backend.clear_session(); keras.utils.set_random_seed(0)
        try:
            m = fn()
        except Exception as e:  # p. ej. sin internet para bajar los pesos
            print(f"{name}: no se pudo construir -> {e}")
            continue
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        tm = EpochTimer()
        m.fit(batches("train", True, 0), validation_data=batches("val", False), epochs=3, callbacks=[tm], verbose=0)
        timing[name], params[name] = float(np.median(tm.times[1:])), int(m.count_params())
        print(f"{name}: {params[name]:,} parámetros | {timing[name]:.1f} s/época (mediana de épocas 2-3)")
display(pd.DataFrame({"s/época": timing, "parámetros": params}).round(2))

## 9. Proyección del presupuesto y tablero de decisión

$$T_{\text{total}} \approx \sum_{m=1}^{3} t_m \times n_{\text{semillas}} \times n_{\text{épocas}}$$

Aquí $t_m$ es el tiempo por época del modelo $m$. Es una **cota superior**: con early stopping se corren menos épocas. Si se suma búsqueda de hiperparámetros (HPO), multiplica por el número de configuraciones probadas.

In [ ]:
b = CFG["budget"]
if len(timing) >= 3:
    total_h = sum(timing.values()) * b["seeds"] * b["epochs"] / 3600
    base_txt = "suma de los 3 modelos cronometrados"
else:
    total_h = timing["cnn_pequena"] * b["models"] * b["seeds"] * b["epochs"] / 3600
    base_txt = "supone que los 3 modelos cuestan como la CNN"

def sem(v, verde, rojo, mayor_es_mejor=False):
    if mayor_es_mejor:
        return "VERDE" if v >= verde else ("ROJO" if v < rojo else "AMARILLO")
    return "VERDE" if v <= verde else ("ROJO" if v > rojo else "AMARILLO")

f1 = np.array([r["val"]["macro_f1"] for r in runs])
f1_mean = float(f1.mean())
f1_sd = float(f1.std(ddof=1)) if len(f1) > 1 else float("nan")
tablero = pd.DataFrame([
    ["Tiempo mediano por época CNN (s)", timing["cnn_pequena"], sem(timing["cnn_pequena"], 8, 12)],
    [f"Proyección presupuesto (h) [{base_txt}]", total_h, sem(total_h, 1.5, 2.0)],
    ["Macro-F1 val CNN (media de semillas)", f1_mean, sem(f1_mean, 0.50, 0.30, mayor_es_mejor=True)],
    ["Ventaja vs. mayoritario (Δ macro-F1)", f1_mean - dummy["most_frequent"]["macro_f1"], "info"],
    ["Margen de mejora para E3 (macro-F1 < 0.90)", 1 - f1_mean, "VERDE" if f1_mean < 0.90 else "ROJO"],
    ["Casi-duplicados entre clases", n_cross, "VERDE" if n_cross == 0 else ("AMARILLO" if n_cross <= 5 else "ROJO")],
    ["Archivos corruptos", len(bad), "VERDE" if len(bad) == 0 else "AMARILLO"],
], columns=["criterio", "valor", "semáforo"])
display(tablero.round(3))
if len(f1) > 1:
    print(f"Macro-F1 val: media {f1_mean:.3f} | DE {f1_sd:.3f} | CV {f1_sd / f1_mean:.1%}  (n = {len(f1)} semillas)")
else:
    print("Una sola semilla: sin estimación de varianza. Para el preregistro usa seeds = [0, 1, 2, 3, 4].")

summary = {"run_id": RUN_ID, "device": DEVICE, "config": CFG,
           "n": {p: int((split["split"] == p).sum()) for p in ["train", "val", "test"]},
           "zip_sha256": zip_sha, "test_fingerprint": test_fp,
           "data_quality": {"corrupt": int(len(bad)), "exact_dups": n_exact,
                            "near_dup_pairs": len(pairs), "cross_class_pairs": n_cross},
           "dummy_val": dummy, "cnn_runs": runs,
           "cnn_val_macro_f1": {"mean": f1_mean, "sd": None if np.isnan(f1_sd) else f1_sd},
           "timing_s_per_epoch": timing, "params": params, "projection_h": total_h,
           "notebook_time_s": time.perf_counter() - T0_NOTEBOOK}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
tablero.to_csv(OUT / "tablero_decision.csv", index=False)
print(f"Tiempo total del notebook: {summary['notebook_time_s'] / 60:.1f} min | guardado en {OUT.relative_to(ROOT)}")

## 10. Notas y decisión (completar a mano)

- **Máquina / dispositivo:**
- **Lo que observé en las curvas (¿overfitting? ¿underfitting?):**
- **Clases que más se confunden y posible causa:**
- **¿Algún casi-duplicado real o falso positivo?**
- **Decisión:** ☐ Seguimos con TrashNet ☐ Ajustamos (resolución / semillas / modelos) ☐ Cambiamos a plan B
- **Justificación (cita el tablero):**
- **Siguiente paso:**